# Eksperyment: Model zbiorczy / Enriched

## Cel
Połączyć dwa wygrywające (na pojedynczym teście) zestawy cech: **surface_speed (3) + fatigue (6)**.
Pytanie: czy zyski się sumują, czy uderza nadmiar cech (curse of dimensionality)?

In [1]:
import sys, io, contextlib, runpy
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import os
# --- Colab: pobiera projekt z GitHuba; lokalnie ten blok jest pomijany ---
# PO UTWORZENIU repo podmien adres ponizej na swoj:
_REPO = "https://github.com/StanislawKarwala/TennisPredictionModel.git"
if "google.colab" in sys.modules and not Path("../src/tennis_model.py").exists():
    import subprocess
    subprocess.run(["pip", "install", "-q", "xgboost"])
    subprocess.run(["git", "clone", "-q", _REPO, "/content/tenis"])
    os.chdir("/content/tenis/notebooks")
sys.path.insert(0, str(Path("../src").resolve()))
from tennis_model_surface_speed import build_court_pace_lookup, court_pace_index
from tennis_model_fatigue import compute_fatigue_for_2024
from tennis_model_enriched import SPEED_FEATURES, FATIGUE_FEATURES, NEW_FEATURES, data_file, TARGET_YEAR
print("Nowe cechy:", len(NEW_FEATURES), "=", len(SPEED_FEATURES), "speed +", len(FATIGUE_FEATURES), "fatigue")

Nowe cechy: 9 = 3 speed + 6 fatigue


## 1. Reuse baseline pipeline

In [2]:
BASE = Path("../src/tennis_model.py").resolve()
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ns = runpy.run_path(str(BASE))
symmetrize_data = ns["symmetrize_data"]
compute_symmetric_match_evaluation = ns["compute_symmetric_match_evaluation"]
evaluate_calibration_quality = ns["evaluate_calibration_quality"]
baseline_search = ns["search"]
RANDOM_STATE = ns["RANDOM_STATE"]
base_features = list(ns["features"]); cols_base = list(ns["cols_base"])
df_train_raw = ns["df_train_raw"].copy(); df_val_raw = ns["df_val_raw"].copy(); df_test_raw = ns["df_test_raw"].copy()
baseline_val_acc = float(ns["val_acc"]); baseline_test_acc = float(ns["test_acc"]); baseline_match_acc = float(ns["match_accuracy"])
print(f"Baseline: val={baseline_val_acc:.4f}  test={baseline_test_acc:.4f}  match={baseline_match_acc:.4f}  (cech: {len(base_features)})")

Baseline: val=0.6106  test=0.6604  match=0.6566  (cech: 40)


## 2. Budowa kontekstu: court_pace + fatigue (wyrównane do roku docelowego)

In [3]:
full_target = pd.read_csv(data_file(TARGET_YEAR))
full_target["tourney_date"] = pd.to_datetime(full_target["tourney_date"], format="%Y%m%d")
full_target = full_target.sort_values(["tourney_date", "match_num"]).reset_index(drop=True)
full_target_base = full_target[cols_base + ["tourney_id", "minutes"]].dropna(subset=cols_base).reset_index(drop=True)
n_train, n_val, n_test = len(df_train_raw), len(df_val_raw), len(df_test_raw)
assert len(full_target_base) == n_train + n_val + n_test
lookup = build_court_pace_lookup()
cpi = np.array([court_pace_index(t, s, lookup) for t, s in zip(full_target_base["tourney_id"], full_target_base["surface"])])
fat = compute_fatigue_for_2024(full_target_base)
context = pd.DataFrame({"court_pace_index": cpi,
    "w_rest_days": fat["w_rest_days"].to_numpy(), "l_rest_days": fat["l_rest_days"].to_numpy(),
    "w_tourney_minutes": fat["w_tourney_minutes"].to_numpy(), "l_tourney_minutes": fat["l_tourney_minutes"].to_numpy()})
ctx_train = context.iloc[:n_train].reset_index(drop=True)
ctx_val = context.iloc[n_train:n_train+n_val].reset_index(drop=True)
ctx_test = context.iloc[n_train+n_val:].reset_index(drop=True)
print("Kontekst:", list(context.columns))

Kontekst: ['court_pace_index', 'w_rest_days', 'l_rest_days', 'w_tourney_minutes', 'l_tourney_minutes']


## 3. Doklejenie + symetryzacja + interakcje serve x speed + fatigue p1/p2

In [4]:
def attach(df_raw, ctx):
    df_raw = df_raw.copy().reset_index(drop=True)
    for col in context.columns: df_raw[col] = ctx[col].to_numpy()
    return df_raw
df_train_raw = attach(df_train_raw, ctx_train); df_val_raw = attach(df_val_raw, ctx_val); df_test_raw = attach(df_test_raw, ctx_test)
raw_ctx = ["match_id"] + list(context.columns)
def build_split(df_raw, shuffle):
    sym = symmetrize_data(df_raw, shuffle=shuffle).merge(df_raw[raw_ctx], on="match_id", how="left", validate="many_to_one")
    w = (sym["y"] == 1).to_numpy()
    sym["ace_speed_diff"] = (sym["p1_ace_rate"]-sym["p2_ace_rate"])*sym["court_pace_index"]
    sym["first_won_speed_diff"] = (sym["p1_first_won_pct"]-sym["p2_first_won_pct"])*sym["court_pace_index"]
    sym["p1_rest_days"]=np.where(w,sym["w_rest_days"],sym["l_rest_days"]); sym["p2_rest_days"]=np.where(w,sym["l_rest_days"],sym["w_rest_days"])
    sym["p1_tourney_minutes"]=np.where(w,sym["w_tourney_minutes"],sym["l_tourney_minutes"]); sym["p2_tourney_minutes"]=np.where(w,sym["l_tourney_minutes"],sym["w_tourney_minutes"])
    sym["rest_days_diff"]=sym["p1_rest_days"]-sym["p2_rest_days"]; sym["tourney_minutes_diff"]=sym["p1_tourney_minutes"]-sym["p2_tourney_minutes"]
    return sym
train_data=build_split(df_train_raw,True); test_data=build_split(df_test_raw,True); val_data=build_split(df_val_raw,True)
features = base_features + NEW_FEATURES
print("Cech razem:", len(features))

Cech razem: 49


## 4. Trening RF + ewaluacja

In [5]:
X_train,y_train=train_data[features],train_data["y"]; X_val,y_val=val_data[features],val_data["y"]; X_test,y_test=test_data[features],test_data["y"]
best_rf = RandomForestClassifier(**baseline_search.best_params_, n_jobs=-1, random_state=RANDOM_STATE).fit(X_train,y_train)
val_acc=float(accuracy_score(y_val,best_rf.predict(X_val))); test_acc=float(accuracy_score(y_test,best_rf.predict(X_test)))
proba=best_rf.predict_proba(X_test)[:,1]; test_data["p1_win_probability"]=proba
_,match_acc=compute_symmetric_match_evaluation(test_data); q=evaluate_calibration_quality(y_test.to_numpy(),proba)
print(f"baseline           match={baseline_match_acc:.4f}")
print(f"+speed+fatigue     match={match_acc:.4f}  Brier={q['brier_score']:.4f}  DELTA={match_acc-baseline_match_acc:+.4f}")

baseline           match=0.6566
+speed+fatigue     match=0.6585  Brier=0.2178  DELTA=+0.0019


## Wnioski
Połączenie prędkości kortu i zmęczenia (9 cech naraz) nie sumuje się do realnego zysku. Na walidacji przez 6 sezonów wychodzi +0,20 p.p. (McNemar p = 0,66), czyli nieistotnie. Dobra wiadomość jest taka, że dołożenie tylu cech niczego nie psuje — ale też nic nie wnosi. Znów ten sam sufit ~65%.